# Topic 22 - Embeddings

Companion notebook for `embeddings.py`. Text becomes a dense vector; meaning becomes geometry: similar meanings land nearby regardless of wording, dimensions and normalisation are knobs you can *see*, and a golden-set harness shows why benchmark leaders are not automatically your domain's winner.

All offline: a PPMI co-occurrence embedder (meaning from 'the company a word keeps') and a deterministic hashing embedder (mechanics only).

## Learning objectives

- see the topic's mechanics running offline, cell by cell
- connect each demo cell to the pytest suite that pins it
- walk away with numbers you can quote from memory

## 0. Import the core

In [1]:
from embeddings import (
    CooccurrenceEmbedder, HashingEmbedder,
    contrastive_score, cosine_similarity, l2_normalize,
    most_similar, neighbor_stability, storage_bytes, tokenize, truncate,
)


## 1. Meaning as geometry: the co-occurrence embedder

Words that keep the same company get vectors pointing the same way. This is the count-based origin story of Word2Vec (Alammar's illustrated guide).

In [2]:
emb = CooccurrenceEmbedder(
    [
        "the dog is loyal and friendly", "a loyal dog protects the family",
        "a faithful dog guards the house", "the playful puppy wags its tail",
        "the puppy chased the ball happily", "puppies love to play and run",
        "dogs bark at strangers loudly", "a good dog sits on command",
        "every puppy grows into a young dog", "the puppy is a loyal young dog",
        "a playful puppy becomes a loyal family dog", "the big dog watches over the little puppy",
        "the stock market rallied on strong earnings", "investors bought shares before the rise",
        "interest rates rose and lending slowed", "the bank lent money to small businesses",
        "bond yields climbed after the report", "the fund bought bonds and shares",
        "markets fell as inflation climbed", "traders watched the market closely",
    ],
    window=2,
)
print(f"vocabulary size (== dimensions): {emb.dimensions}")
print(f"vector('dog') has {sum(v != 0 for v in emb.embed_word('dog'))} non-zero axes")
for name, vec in most_similar(emb.embed_word("dog"), dict(emb.vocab))[:6]:
    print(f"  dog ~ {name:10s}  cos = {vec:.3f}")


vocabulary size (== dimensions): 77
vector('dog') has 15 non-zero axes
  dog ~ dog         cos = 1.000
  dog ~ a           cos = 0.509
  dog ~ family      cos = 0.414
  dog ~ watches     cos = 0.412
  dog ~ loyal       cos = 0.366
  dog ~ protects    cos = 0.325


## 2. Similar meaning lands nearby, regardless of wording

Paraphrase invariance is the property you buy: no shared words + same meaning still beats shared topic-glue words.

In [3]:
q = emb.embed_text("the puppy is loyal")
same_meaning = emb.embed_text("a dog is faithful")
different_topic = emb.embed_text("the stock market rallied")
print(f"cos(query, faithful dog)   = {cosine_similarity(q, same_meaning):.3f}  (no shared content words)")
print(f"cos(query, stock rally)    = {cosine_similarity(q, different_topic):.3f}")
assert cosine_similarity(q, same_meaning) > cosine_similarity(q, different_topic)
print("\ngeometry grouped them without a single synonym in common")


cos(query, faithful dog)   = 0.643  (no shared content words)
cos(query, stock rally)    = 0.338

geometry grouped them without a single synonym in common


## 3. A letter-machine is NOT a model

The hashing embedder builds vectors deterministically from tokens and character 3-grams. Shared letters pull texts together (run/running) - but it cannot know a puppy is a pet when the strings share nothing.

Then the two golden-set contrastive scores, on two domains - the benchmark-caveat harness in miniature.

In [4]:
letters = HashingEmbedder(dimensions=64)
print(f"cos('run', 'running')  = {cosine_similarity(letters.embed('run'), letters.embed('running')):.3f}   (shared trigrams)")
print(f"cos('run', 'quantum') = {cosine_similarity(letters.embed('run'), letters.embed('quantum')):.3f}")
print()
gold_query = "loyal dog guards the family"
pos = ["a faithful dog protects the house", "the puppy stayed close"]
neg = ["the stock market rallied", "bond yields climbed"]
semantic = contrastive_score(emb.embed_text, gold_query, pos, neg)
hashing = contrastive_score(letters.embed, gold_query, pos, neg)
print(f"golden-set contrastive score, meaning domain:")
print(f"  co-occurrence embedder: {semantic:+.3f}   <- wins on meaning\n"
      f"  hashing embedder:       {hashing:+.3f}")
print()
stringy_query = "running race results"
stringy_pos = ["the runner raced fast", "racing results"]
stringy_neg = ["loyal dog sleeps", "cat purrs quietly"]
stringy_hashing = contrastive_score(letters.embed, stringy_query, stringy_pos, stringy_neg)
stringy_semantic = contrastive_score(emb.embed_text, stringy_query, stringy_pos, stringy_neg)
print(f"stringy domain (letter overlap wins):")
print(f"  hashing embedder:       {stringy_hashing:+.3f}   <- wins on letters\n"
      f"  co-occurrence embedder: {stringy_semantic:+.3f}")
print("\nbenchmark leaders are per-corpus: measure on YOUR pairs")


cos('run', 'running')  = 0.228   (shared trigrams)
cos('run', 'quantum') = -0.252

golden-set contrastive score, meaning domain:
  co-occurrence embedder: +0.436   <- wins on meaning
  hashing embedder:       +0.172

stringy domain (letter overlap wins):
  hashing embedder:       +0.356   <- wins on letters
  co-occurrence embedder: +0.000

benchmark leaders are per-corpus: measure on YOUR pairs


## 4. Dimensions: cost, truncation, and the unit sphere

pgvector accounting: a float32 vector takes 4 x dims + 8 bytes. Matryoshka-style models keep quality when you slice trailing dimensions (then re-normalise!).

In [5]:
for dims in (384, 768, 1536, 3072):
    bytes_ = storage_bytes(dims)
    print(f"  {dims:5d} dims -> {bytes_:6d} bytes/vector -> {bytes_ * 1_000_000 / 1e9:.2f} GB per 1M docs")
print()
letters128 = HashingEmbedder(dimensions=128)
candidates = {f"doc-{i}": letters128.embed(f"quarterly report {i} revenue margin") for i in range(12)}
query = letters128.embed("quarterly revenue report")
for cut in (128, 64, 32, 16):
    kept = neighbor_stability(query, candidates, top_k=3, dimensions=cut)
    print(f"  truncate 128 -> {cut:3d} dims: {kept}/3 top neighbours survive")


    384 dims ->   1544 bytes/vector -> 1.54 GB per 1M docs
    768 dims ->   3080 bytes/vector -> 3.08 GB per 1M docs
   1536 dims ->   6152 bytes/vector -> 6.15 GB per 1M docs
   3072 dims ->  12296 bytes/vector -> 12.30 GB per 1M docs

  truncate 128 -> 128 dims: 3/3 top neighbours survive
  truncate 128 ->  64 dims: 1/3 top neighbours survive
  truncate 128 ->  32 dims: 2/3 top neighbours survive
  truncate 128 ->  16 dims: 1/3 top neighbours survive


## 5. The unit-sphere invariant (normalisation)

Embedding APIs already L2-normalise. On the unit sphere cosine == dot, and ranking by cosine equals ranking by Euclidean distance.

In [6]:
for text in ["loyal dog", "", "puppy plays every day"]:
    v = letters.embed(text)
    norm = sum(x * x for x in v) ** 0.5
    print(f"  ||embed({text!r:>22})|| = {norm:.6f}")
v1, v2 = emb.embed_text("the dog is loyal"), emb.embed_text("a loyal dog")
print(f"\ncos(same sentence, reordered) = {cosine_similarity(v1, v2):.3f}")


  ||embed(           'loyal dog')|| = 1.000000
  ||embed(                    '')|| = 0.000000
  ||embed('puppy plays every day')|| = 1.000000

cos(same sentence, reordered) = 0.816


## 6. Run the tests

In [7]:
import subprocess
import sys

result = subprocess.run([sys.executable, "-m", "pytest", "-q"], capture_output=True, text=True)
print(result.stdout[-1500:])
assert "failed" not in result.stdout, "pytest failures!"
print("\noffline tests passed")


...............................                                          [100%]
31 passed in 0.06s


offline tests passed


## 7. What to notice

- geometry came from co-occurrence statistics, no neural net anywhere
- the hashing embedder is fast and deterministic but knows letters, not meaning
- golden-set contrastive scores are how you pick a model for YOUR domain, not leaderboards
- dimensions are a memory bill (4*d+8 bytes/row) and a Matryoshka trade-off
- everything sorts on the unit sphere: normalise once, compare forever
- the pytest cell ran 31 tests and prints 'offline tests passed'